In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

import lightgbm as lgb

SEED = 42
np.random.seed(SEED)

In [2]:
train = pd.read_csv('/kaggle/input/first-competition-exhibition/train.csv')
test  = pd.read_csv('/kaggle/input/first-competition-exhibition/test.csv')

test_ids = test['id']

train = train.drop(columns=['id', 'Row#'])
test  = test.drop(columns=['id', 'Row#'])

In [3]:
def create_features(df, kmeans=None, scaler=None, fit=False):
    data = df.copy()

    # ---- Bees ----
    data['total_bees'] = (
        data['honeybee'] +
        data['bumbles'] +
        data['andrena'] +
        data['osmia']
    )

    data['bees_per_clone'] = data['total_bees'] / (data['clonesize'] + 1e-6)
    data['osmia_honeybee_inter'] = data['osmia'] * data['honeybee']

    # ---- Temperature ----
    data['temp_range'] = (
        data['MaxOfUpperTRange'] -
        data['MinOfLowerTRange']
    )

    data['avg_temp'] = (
        data['AverageOfUpperTRange'] +
        data['AverageOfLowerTRange']
    ) / 2

    data['temp_x_rain'] = data['avg_temp'] * data['RainingDays']

    # ---- Non-linear ----
    for col in ['clonesize', 'fruitmass', 'seeds']:
        data[f'log_{col}'] = np.log1p(data[col])

    data['fruit_seed_ratio'] = data['fruitmass'] / (data['seeds'] + 1e-6)

    # ---- Single Clustering (🔥 فقط یکی) ----
    cluster_cols = ['clonesize', 'avg_temp', 'RainingDays', 'fruitmass']

    if fit:
        scaler = StandardScaler()
        scaled = scaler.fit_transform(data[cluster_cols])

        kmeans = KMeans(
            n_clusters=7,          # 🔥 sweet spot
            random_state=SEED,
            n_init=40
        )
        data['cluster'] = kmeans.fit_predict(scaled)
    else:
        scaled = scaler.transform(data[cluster_cols])
        data['cluster'] = kmeans.predict(scaled)

    return data, kmeans, scaler


train_fe, kmeans, scaler = create_features(train, fit=True)
test_fe, _, _ = create_features(test, kmeans=kmeans, scaler=scaler)

In [4]:
X = train_fe.drop(columns=['yield'])
y = train_fe['yield']

In [5]:
params = {
    'objective': 'regression_l1',
    'metric': 'mae',

    'learning_rate': 0.022,
    'num_leaves': 72,          # 🔥 کنترل leaf
    'min_data_in_leaf': 45,    # 🔥 جلوگیری از overfit

    'feature_fraction': 0.88,
    'bagging_fraction': 0.88,
    'bagging_freq': 1,

    'lambda_l1': 0.6,
    'lambda_l2': 1.2,

    'boosting': 'gbdt',

    'max_depth': -1,
    'verbosity': -1,
    'seed': SEED,

    'device': 'gpu',
    'gpu_platform_id': 0,
    'gpu_device_id': 0
}

# =====================
# CV
# =====================
kf = KFold(n_splits=10, shuffle=True, random_state=SEED)

oof = np.zeros(len(X))
test_preds = np.zeros(len(test_fe))
maes = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X), 1):
    print(f'Fold {fold}/10')

    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    train_set = lgb.Dataset(X_tr, y_tr)
    val_set   = lgb.Dataset(X_val, y_val)

    model = lgb.train(
        params,
        train_set,
        num_boost_round=5200,
        valid_sets=[val_set],
        callbacks=[
            lgb.early_stopping(300, verbose=False),
            lgb.log_evaluation(0)
        ]
    )

    val_pred = model.predict(X_val)
    oof[val_idx] = val_pred

    fold_mae = mean_absolute_error(y_val, val_pred)
    maes.append(fold_mae)

    test_preds += model.predict(test_fe) / kf.n_splits

print('\n🔥 OOF MAE:', np.mean(maes), '±', np.std(maes))

Fold 1/10


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


Fold 2/10
Fold 3/10
Fold 4/10
Fold 5/10
Fold 6/10
Fold 7/10
Fold 8/10
Fold 9/10
Fold 10/10

🔥 OOF MAE: 245.3759357524801 ± 6.086919135806937


In [6]:
final_test_preds = np.clip(
    test_preds,
    train['yield'].min(),
    train['yield'].max()
)

submission = pd.DataFrame({
    'id': test_ids,
    'yield': final_test_preds
})

submission.to_csv('submission.csv', index=False)
submission.head()

,id,yield
0,15000,7485.490542
1,15001,5903.252552
2,15002,6480.757762
3,15003,4683.176963
4,15004,5897.682983
